# Four Multi-Agent Orchestration Patterns in Python

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/agents/multi_agent_orchestration.ipynb)

Companion notebook to [the post](https://sesen.ai/blog/multi-agent-orchestration-python).

One job, six sections, five ways to get it done. Every pattern returns the same
object: a list of `Call` records forming a dependency graph. Token spend is a sum
over the graph, wall-clock is the graph's critical path under a concurrency cap,
and accuracy is measured against ground truth.

Nothing here calls a language model. There is no API key, no network and no cost,
and the whole notebook runs in about a second.

## 1. The corpus

Eighteen documents. Six carry the facts the briefing needs. The other twelve are
the ordinary contents of a company wiki, including two earlier incidents whose
postmortems use the same vocabulary as this one.

In [ ]:
import re
from collections import Counter
from dataclasses import dataclass

INCIDENT = "INC-4471"

@dataclass(frozen=True)
class Doc:
    doc_id: str
    domain: str
    text: str

    @property
    def tokens(self) -> int:
        return count_tokens(self.text)


_DOCS = [
    # ---- the six documents that carry the briefing's facts
    Doc(
        "pager/INC-4471",
        "infra",
        "Pager record INC-4471. Alert export_lag_seconds fired at 02:14 UTC on "
        "2026-08-27. Primary on-call Priya Raman acknowledged at 02:19 and "
        "escalated to the data platform team at 02:31. Mitigation applied at "
        "03:44. Alert cleared and the incident was resolved at 04:02 UTC.",
    ),
    Doc(
        "postmortem/INC-4471-draft",
        "infra",
        "Draft postmortem for INC-4471. Root cause: migration 0142 removed the "
        "NOT NULL default on orders.export_state. Rows written after the "
        "migration carried a null export_state and the exporter upsert rejected "
        "every one of them. The failure was silent because the exporter counts "
        "rejected rows as skipped rather than failed.",
    ),
    Doc(
        "runbook/export-recovery",
        "infra",
        "Runbook, export recovery. To recover a stalled export: roll back the "
        "offending migration, then re-run the backfill job with "
        "--from-checkpoint. Never truncate the export queue. The INC-4471 "
        "recovery rolled back migration 0142 and backfilled 1884 orders.",
    ),
    Doc(
        "billing/affected-accounts",
        "customer",
        "Impact query for INC-4471. 212 accounts held at least one order stuck in "
        "the export queue, 1884 orders in total. Three enterprise tenants are in "
        "scope: Halden Freight, Corio Retail and Lumen Health. No billing records "
        "were altered and no invoices were reissued.",
    ),
    Doc(
        "comms/INC-4471-status-note",
        "customer",
        "Customer notification for INC-4471. The status page carried a partial "
        "degradation notice for order export from 02:40 to 04:20 UTC. Named "
        "account managers contacted the three enterprise tenants directly, "
        "inside the four-hour commitment in the support agreement.",
    ),
    Doc(
        "risk/change-register-0142",
        "risk",
        "Change register entry for migration 0142. Shipped without a backfill dry "
        "run. Four further migrations are queued under the same exemption, 0143 "
        "to 0146. The dry-run gate was made optional in June to clear a release "
        "backlog and has not been restored.",
    ),
    # ---- twelve decoys: an ordinary wiki, all high overlap, none with the facts
    Doc(
        "wiki/export-pipeline-architecture",
        "infra",
        "Export pipeline architecture. The pipeline reads the orders table, "
        "applies the export_state machine, and writes to each tenant sink. A "
        "nightly backfill job replays anything the stream missed. Migration "
        "history for the orders table lives in the schema registry. Risk controls "
        "for the pipeline are recorded in the change register.",
    ),
    Doc(
        "postmortem/INC-4102",
        "infra",
        "Postmortem for INC-4102, 2026-06-11. Root cause: a migration removed an "
        "index used by the exporter upsert and export lag rose until the job was "
        "rolled back. The alert fired, on-call acknowledged, the team escalated, "
        "and the incident was resolved the same night. A backfill recovered the "
        "affected orders.",
    ),
    Doc(
        "postmortem/INC-3980",
        "infra",
        "Postmortem for INC-3980, 2026-04-02. Root cause: the export queue filled "
        "after a tenant sink stopped acknowledging writes. Export lag rose, the "
        "alert fired, and on-call escalated to the data platform team. Mitigation "
        "was to drain the queue and re-run the backfill job.",
    ),
    Doc(
        "runbook/schema-migrations",
        "infra",
        "Runbook, schema migrations. Every migration is written with an up and a "
        "down step. Roll back with the down step, never by hand. Run the backfill "
        "dry run before shipping a migration that changes a default or a NOT NULL "
        "constraint on a large table.",
    ),
    Doc(
        "runbook/on-call",
        "infra",
        "Runbook, on-call. The primary acknowledges inside ten minutes and "
        "escalates to the owning team inside thirty. Record the acknowledgement "
        "time, the escalation time and the resolution time in the pager record "
        "for every incident.",
    ),
    Doc(
        "wiki/alerting-thresholds",
        "infra",
        "Alerting thresholds. The export_lag_seconds alert fires when lag exceeds "
        "900 seconds for five consecutive minutes. It was tuned in March after a "
        "run of false pages. Alert history and firing times are held in the pager "
        "system, not here.",
    ),
    Doc(
        "tickets/4471",
        "customer",
        "Support ticket 4471, opened by an external customer on 2026-08-27. Order "
        "88213 failed to export. The customer reports the order has been stuck "
        "since the morning, asks when the export will complete, and asks whether "
        "other orders on their account are affected.",
    ),
    Doc(
        "comms/incident-templates",
        "customer",
        "Customer notification templates. A partial degradation notice names the "
        "affected surface and the window. Enterprise tenants with a support "
        "agreement are contacted directly by their named account manager inside "
        "four hours. Do not publish a root cause before the postmortem is agreed.",
    ),
    Doc(
        "billing/credit-policy",
        "customer",
        "Service credit policy. Accounts on an enterprise agreement may claim a "
        "credit when an incident breaches the monthly availability target. Export "
        "delays under six hours do not breach the target. Credits are applied to "
        "the following invoice, never refunded.",
    ),
    Doc(
        "wiki/tenant-directory",
        "customer",
        "Tenant directory. Enterprise tenants and their named account managers: "
        "Halden Freight, Corio Retail, Lumen Health, Verrall Logistics and "
        "Ashcombe Foods. Each entry lists the support agreement tier and the "
        "escalation contact.",
    ),
    Doc(
        "risk/change-register-0138",
        "risk",
        "Change register entry for migration 0138. Shipped in May with a backfill "
        "dry run recorded and signed off by the data platform team. No exemption "
        "was claimed. Closed with no follow-up actions.",
    ),
    Doc(
        "risk/audit-actions-q2",
        "risk",
        "Quarterly audit actions. Four control gates were reviewed. Two were "
        "recorded as effective. The change management gate was recorded as "
        "partially effective, with a follow-up owner assigned and a review date "
        "in September.",
    ),
]

CORPUS: dict[str, Doc] = {doc.doc_id: doc for doc in _DOCS}

## 2. The job

Six sections. Each names its domain, the question it answers, and the one
document that carries its facts. `source` is the ground truth.

In [ ]:
@dataclass(frozen=True)
class Section:
    name: str
    domain: str
    brief: str
    source: str  # the document that carries this section's facts
    answer: str  # what a correct section says


SECTIONS: list[Section] = [
    Section(
        "timeline",
        "infra",
        "timeline of the incident from the first alert through to resolution",
        "pager/INC-4471",
        "Alert 02:14 UTC, acknowledged 02:19, escalated 02:31, resolved 04:02 UTC.",
    ),
    Section(
        "root_cause",
        "infra",
        "root cause of the export failure and why it went unnoticed",
        "postmortem/INC-4471-draft",
        "Migration 0142 dropped the NOT NULL default on orders.export_state, the "
        "exporter upsert rejected every row written afterwards, and the "
        "rejections were counted as skips.",
    ),
    Section(
        "mitigation",
        "infra",
        "mitigation applied and the recovery steps that were run",
        "runbook/export-recovery",
        "Rolled back migration 0142, then re-ran the backfill from checkpoint, "
        "recovering 1884 orders.",
    ),
    Section(
        "blast_radius",
        "customer",
        "how many accounts and orders were affected, and which enterprise tenants",
        "billing/affected-accounts",
        "212 accounts and 1884 orders, including Halden Freight, Corio Retail and "
        "Lumen Health.",
    ),
    Section(
        "comms",
        "customer",
        "what customers were told about this incident and when they were told",
        "comms/INC-4471-status-note",
        "Status page degradation notice 02:40 to 04:20 UTC; account managers "
        "contacted the three enterprise tenants inside the four-hour commitment.",
    ),
    Section(
        "regression_risk",
        "risk",
        "risk of the same failure recurring and what still has to change",
        "risk/change-register-0142",
        "Migrations 0143 to 0146 are queued under the same waived dry-run gate, "
        "optional since June.",
    ),
]

SECTION_BY_NAME = {section.name: section for section in SECTIONS}

JOB = (
    f"Write the incident briefing for {INCIDENT}, six sections: "
    + ", ".join(section.name for section in SECTIONS)
    + "."
)

# A hand-built map from section to source documents, two each. Not an oracle:
# it names candidate documents, and one of the two is wrong for four of the six
# sections. Building it is an afternoon with the wiki, not a modelling advance.
SECTION_INDEX: dict[str, tuple[str, ...]] = {
    "timeline": ("pager/INC-4471", "runbook/on-call"),
    "root_cause": ("postmortem/INC-4471-draft", "runbook/schema-migrations"),
    "mitigation": ("runbook/export-recovery", "pager/INC-4471"),
    "blast_radius": ("billing/affected-accounts", "wiki/tenant-directory"),
    "comms": ("comms/INC-4471-status-note", "comms/incident-templates"),
    "regression_risk": ("risk/change-register-0142", "risk/audit-actions-q2"),
}

## 3. Tokens and retrieval

Token counting approximates a byte-pair count without a tokeniser dependency.
Retrieval is length-normalised token overlap, the same instinct behind BM25's
length normalisation.

In [ ]:
_TOKEN = re.compile(r"\w+|[^\w\s]")


def count_tokens(text: str) -> int:
    """Approximate a byte-pair token count with no tokeniser dependency.

    Words of four characters or fewer cost one token, longer words one per four
    characters, punctuation one. Every comparison in the post is a ratio between
    patterns running on the same text, so the approximation cancels.
    """
    return sum(max(1, -(-len(piece) // 4)) for piece in _TOKEN.findall(text))

_WORD = re.compile(r"[a-z0-9_]+")
_STOP = {
    "the", "and", "for", "with", "that", "from", "was", "were", "are", "its",
    "this", "what", "when", "any", "not", "has", "had", "have", "been", "into",
    "of", "to", "in", "on", "at", "a", "an", "it", "is", "be", "by", "or",
    "every", "each", "no", "after", "before", "than", "through", "under",
    "their", "they", "them", "which", "who", "how", "still", "same", "other",
    "will", "may", "do", "does", "did", "one", "two", "three", "four", "five",
}


def _bag(text: str) -> set[str]:
    return {word for word in _WORD.findall(text.lower()) if word not in _STOP}


def _score(query: str, doc: Doc) -> float:
    """Query-document overlap, normalised by the square root of document length.

    The same instinct as BM25's length normalisation: a long document should not
    win on raw overlap.
    """
    doc_bag = _bag(doc.text)
    if not doc_bag:
        return 0.0
    return len(_bag(query) & doc_bag) / len(doc_bag) ** 0.5


def keyword(query: str, budget: int, domain: str | None = None) -> list[str]:
    """Top-`budget` documents by score, ties broken by document id."""
    pool = [doc for doc in CORPUS.values() if domain is None or doc.domain == domain]
    ranked = sorted(pool, key=lambda doc: (-_score(query, doc), doc.doc_id))
    return [doc.doc_id for doc in ranked[:budget]]


RETRIEVERS = ("keyword-2", "keyword-4", "keyword-8", "domain-2", "index")


def retrieve(strategy: str, query: str, section: Section | None) -> list[str]:
    """Run one retrieval strategy.

    `keyword-N` ranks the whole corpus and ignores the schema the brief already
    carries. `domain-N` ranks inside the domain the brief declares. `index`
    reads the hand-built section index.
    """
    if strategy == "index":
        if section is not None:
            return list(SECTION_INDEX[section.name])
        seen: list[str] = []
        for docs in SECTION_INDEX.values():
            seen += [doc for doc in docs if doc not in seen]
        return seen

    name, _, size = strategy.partition("-")
    width = int(size)
    if name != "domain":
        return keyword(query, width)
    if section is not None:
        return keyword(query, width, domain=section.domain)
    out: list[str] = []
    for domain in DOMAINS:
        out += keyword(query, width, domain=domain)
    return out

## 4. Routing

The specialist cards are not hand-written. They are the most frequent terms in
each domain's own documents, which is how such descriptions get written in
practice, and it is why they overlap when the corpora overlap.

In [ ]:
DOMAINS = ("infra", "customer", "risk")


def _domain_vocabulary(domain: str, size: int = 24) -> set[str]:
    """The specialist's description, read off the documents it owns.

    Specialist cards in a real supervisor prompt are written from what the
    specialist's corpus talks about, which is why they overlap when the corpora
    overlap.
    """
    counts: Counter[str] = Counter()
    for doc in CORPUS.values():
        if doc.domain == domain:
            counts.update(_bag(doc.text))
    return {word for word, _ in counts.most_common(size)}


DOMAIN_CARDS = {domain: _domain_vocabulary(domain) for domain in DOMAINS}


def route(section: Section, cards: dict[str, set[str]] | None = None) -> str:
    """Pick a specialist by overlap between the section brief and each card."""
    cards = cards or DOMAIN_CARDS
    brief = _bag(section.brief)
    return sorted(DOMAINS, key=lambda d: (-len(brief & cards[d]), d))[0]

## 5. The scripted model

One rule: the section is right when the document carrying its facts is in the
context, and wrong otherwise. A model cannot report a fact it was never shown.

This buys reproducibility and costs realism in exactly one column. Read the
accuracy figures as a measurement of what each topology does to the context, and
not of what a real model does with the context once it has it. Cost and latency
carry no such caveat: call counts, token counts and the dependency graph are
properties of the shape.

In [ ]:
def answer_section(section: Section, context: list[str]) -> tuple[str, bool]:
    """The scripted model.

    One rule: the section is right when the document carrying its facts is in
    the context, wrong otherwise. A model cannot report a fact it was never
    shown, and when the fact is missing it writes the nearest thing it did see.
    """
    if section.source in context:
        return section.answer, True
    return _ERRORS[section.name], False


# Every wrong answer is fluent, on topic, and drawn from a decoy that retrieval
# did put in the context. None of them is flagged as low confidence.
_ERRORS = {
    "timeline": "The alert fired overnight, on-call acknowledged inside ten "
                "minutes and escalated inside thirty, and the incident was "
                "resolved the same night.",
    "root_cause": "A migration removed an index used by the exporter upsert, "
                  "and export lag rose until the job was rolled back.",
    "mitigation": "The queue was drained and the nightly backfill job re-run to "
                  "recover the affected orders.",
    "blast_radius": "Enterprise tenants on a support agreement were affected, "
                    "including Halden Freight, Corio Retail, Lumen Health, "
                    "Verrall Logistics and Ashcombe Foods.",
    "comms": "A partial degradation notice was published naming the affected "
             "surface and the window, and enterprise tenants were contacted "
             "inside four hours.",
    "regression_risk": "The change management gate was recorded as partially "
                       "effective, with a follow-up owner and a September "
                       "review date.",
}

## 6. Price, speed and the call record

Prices and serving speeds are for a mid-tier hosted model. Change them and every
absolute number moves together, while the ratios between patterns stay put.

In [ ]:
import heapq
from collections import defaultdict
from dataclasses import dataclass, field

# Published rates for a mid-tier hosted model, dollars per million tokens.
PRICE_IN, PRICE_OUT = 3.00, 15.00

# Typical serving figures: time to first token, then a steady output rate, plus
# the prefill cost of the prompt itself.
TTFT = 0.35
SECONDS_PER_OUTPUT_TOKEN = 0.012  # about 83 tokens per second
SECONDS_PER_PROMPT_TOKEN = 0.00008  # about 12,500 tokens per second

SYSTEM = (
    "You are writing one section of an incident briefing for Northwind "
    "Analytics. Use only the documents provided. Answer in one or two "
    "sentences. Do not speculate beyond the documents."
)
SYSTEM_TOKENS = count_tokens(SYSTEM)
JOB_TOKENS = count_tokens(JOB)
ROUTER_PROMPT_TOKENS = 78  # the three specialist cards, plus the instruction


@dataclass
class Call:
    call_id: str
    role: str  # single, worker, router, drafter, judge, reduce
    deps: tuple[str, ...]
    prompt_tokens: int
    output_tokens: int
    section: str | None = None
    context: tuple[str, ...] = ()
    correct: bool | None = None

    @property
    def latency(self) -> float:
        return (
            TTFT
            + self.prompt_tokens * SECONDS_PER_PROMPT_TOKEN
            + self.output_tokens * SECONDS_PER_OUTPUT_TOKEN
        )

    @property
    def cost(self) -> float:
        return (self.prompt_tokens * PRICE_IN + self.output_tokens * PRICE_OUT) / 1e6


@dataclass
class Run:
    pattern: str
    retriever: str
    calls: list[Call] = field(default_factory=list)
    sections: dict[str, bool] = field(default_factory=dict)

    @property
    def prompt_tokens(self) -> int:
        return sum(call.prompt_tokens for call in self.calls)

    @property
    def output_tokens(self) -> int:
        return sum(call.output_tokens for call in self.calls)

    @property
    def tokens(self) -> int:
        return self.prompt_tokens + self.output_tokens

    @property
    def cost(self) -> float:
        return sum(call.cost for call in self.calls)

    @property
    def serial_latency(self) -> float:
        return sum(call.latency for call in self.calls)

    @property
    def correct(self) -> int:
        return sum(self.sections.values())

    def wall_clock(self, concurrency: int | None = None) -> float:
        return schedule(self.calls, concurrency)[1]

## 7. The scheduler

List scheduling over the dependency graph. With no cap the makespan is the
graph's critical path. With a cap it is what an account with a rate limit gets,
which is the number that matters as soon as a fan-out is wider than the limit.

In [ ]:
def schedule(
    calls: list[Call], concurrency: int | None = None
) -> tuple[list[tuple[str, float, float]], float]:
    """List-schedule the dependency graph, returning the timeline and makespan.

    With no cap the makespan is the graph's critical path. With a cap it is what
    an account with a rate limit gets, which is the number that matters as soon
    as a fan-out is wider than the limit.
    """
    by_id = {call.call_id: call for call in calls}
    waiting = {call.call_id: set(call.deps) for call in calls}
    dependents: dict[str, list[str]] = defaultdict(list)
    for call in calls:
        for dep in call.deps:
            dependents[dep].append(call.call_id)

    limit = concurrency or len(calls)
    ready = [(0.0, cid) for cid, deps in waiting.items() if not deps]
    heapq.heapify(ready)
    running: list[tuple[float, str]] = []
    timeline: list[tuple[str, float, float]] = []
    now = 0.0

    while ready or running:
        while ready and len(running) < limit and ready[0][0] <= now:
            release, cid = heapq.heappop(ready)
            end = now + by_id[cid].latency
            timeline.append((cid, now, end))
            heapq.heappush(running, (end, cid))

        if running:
            now, done = heapq.heappop(running)
            for cid in dependents[done]:
                waiting[cid].discard(done)
                if not waiting[cid]:
                    heapq.heappush(ready, (now, cid))
        elif ready:
            now = ready[0][0]

    return timeline, max((end for _, _, end in timeline), default=0.0)

def _doc_tokens(context: list[str]) -> int:
    return sum(CORPUS[doc].tokens for doc in context)


def _worker_call(
    call_id: str,
    section: Section,
    context: list[str],
    deps: tuple[str, ...],
    extra_prompt: int = 0,
    role: str = "worker",
) -> tuple[Call, str, bool]:
    text, correct = answer_section(section, context)
    call = Call(
        call_id=call_id,
        role=role,
        deps=deps,
        prompt_tokens=SYSTEM_TOKENS
        + JOB_TOKENS
        + count_tokens(section.brief)
        + _doc_tokens(context)
        + extra_prompt,
        output_tokens=count_tokens(text),
        section=section.name,
        context=tuple(context),
        correct=correct,
    )
    return call, text, correct


def _reduce_call(call_id: str, deps: tuple[str, ...], texts: list[str]) -> Call:
    """The synthesis call: it reads every section and writes the briefing."""
    body = sum(count_tokens(text) for text in texts)
    return Call(
        call_id=call_id,
        role="reduce",
        deps=deps,
        prompt_tokens=SYSTEM_TOKENS + JOB_TOKENS + body,
        output_tokens=body,
    )


def retrieve_within(strategy: str, section: Section, domain: str) -> list[str]:
    """Retrieve for a specialist that only owns one domain's documents.

    A misrouted section reaches a specialist whose corpus does not hold its
    source, and no retrieval budget inside that corpus can recover it.
    """
    if strategy == "index":
        return [
            doc for doc in SECTION_INDEX[section.name] if CORPUS[doc].domain == domain
        ]
    _, _, size = strategy.partition("-")
    return keyword(section.brief, int(size), domain=domain)

## 8. The five orchestrations

Each returns a `Run`. Read them side by side: the difference between them is
entirely in which calls exist, what goes in each prompt, and which calls wait for
which.

In [ ]:
def run_single(retriever: str) -> Run:
    """One agent, one call. Retrieval runs once, against the whole job."""
    run = Run("single", retriever)
    context = retrieve(retriever, JOB, None)
    texts = []
    for section in SECTIONS:
        text, correct = answer_section(section, context)
        run.sections[section.name] = correct
        texts.append(text)

    run.calls.append(
        Call(
            call_id="single",
            role="single",
            deps=(),
            prompt_tokens=SYSTEM_TOKENS
            + JOB_TOKENS
            + sum(count_tokens(s.brief) for s in SECTIONS)
            + _doc_tokens(context),
            output_tokens=sum(count_tokens(t) for t in texts),
            context=tuple(context),
            correct=all(run.sections.values()),
        )
    )
    return run


def run_sequential(retriever: str) -> Run:
    """One agent, one call per section, each seeing everything written before it.

    The context accumulates: by the last section the prompt carries every
    document retrieved so far and every answer written so far. That is where the
    token bill stops being linear.
    """
    run = Run("sequential", retriever)
    seen: list[str] = []
    written = 0
    previous: tuple[str, ...] = ()

    for index, section in enumerate(SECTIONS):
        for doc in retrieve(retriever, section.brief, section):
            if doc not in seen:
                seen.append(doc)
        call, text, correct = _worker_call(
            f"s{index}", section, list(seen), previous, extra_prompt=written
        )
        written += count_tokens(text)
        run.calls.append(call)
        run.sections[section.name] = correct
        previous = (call.call_id,)
    return run


def run_supervisor(retriever: str) -> Run:
    """A router picks a specialist per section, then that specialist answers.

    The router's own call is cheap. What it costs is a serial hop before every
    worker, and a wrong choice the worker cannot recover from.
    """
    run = Run("supervisor", retriever)
    texts = []

    for index, section in enumerate(SECTIONS):
        run.calls.append(
            Call(
                call_id=f"r{index}",
                role="router",
                deps=(),
                prompt_tokens=SYSTEM_TOKENS
                + count_tokens(section.brief)
                + ROUTER_PROMPT_TOKENS,
                output_tokens=4,
                section=section.name,
            )
        )
        context = retrieve_within(retriever, section, route(section))
        call, text, correct = _worker_call(
            f"w{index}", section, context, (f"r{index}",)
        )
        run.calls.append(call)
        run.sections[section.name] = correct
        texts.append(text)

    run.calls.append(
        _reduce_call("synth", tuple(f"w{i}" for i in range(len(SECTIONS))), texts)
    )
    return run


def run_parallel(retriever: str) -> Run:
    """Every section retrieved and written independently, then one reduce call."""
    run = Run("parallel", retriever)
    texts = []
    for index, section in enumerate(SECTIONS):
        context = retrieve(retriever, section.brief, section)
        call, text, correct = _worker_call(f"p{index}", section, context, ())
        run.calls.append(call)
        run.sections[section.name] = correct
        texts.append(text)

    run.calls.append(
        _reduce_call("reduce", tuple(f"p{i}" for i in range(len(SECTIONS))), texts)
    )
    return run


DIVERSE_PANEL = ("keyword-2", "domain-2", "index")


def run_debate(retriever: str, panel: tuple[str, ...] | None = None) -> Run:
    """Three drafters per section, then a judge takes the majority.

    `panel` sets what the three drafters differ in. Left unset they are three
    copies of one configuration, which is what "run it three times and vote"
    means in practice.
    """
    seats = panel or (retriever, retriever, retriever)
    diverse = len(set(seats)) > 1
    run = Run("debate-diverse" if diverse else "debate", retriever)
    texts = []

    for index, section in enumerate(SECTIONS):
        votes, drafts = [], []
        for seat, strategy in enumerate(seats):
            context = retrieve(strategy, section.brief, section)
            call, text, correct = _worker_call(
                f"d{index}_{seat}", section, context, (), role="drafter"
            )
            run.calls.append(call)
            votes.append(correct)
            drafts.append(text)

        majority = sum(votes) * 2 > len(votes)
        chosen = drafts[votes.index(majority)]
        run.sections[section.name] = majority
        texts.append(chosen)

        run.calls.append(
            Call(
                call_id=f"j{index}",
                role="judge",
                deps=tuple(f"d{index}_{seat}" for seat in range(len(seats))),
                prompt_tokens=SYSTEM_TOKENS
                + count_tokens(section.brief)
                + sum(count_tokens(draft) for draft in drafts),
                output_tokens=count_tokens(chosen),
                section=section.name,
                correct=majority,
            )
        )

    run.calls.append(
        _reduce_call("synth", tuple(f"j{i}" for i in range(len(SECTIONS))), texts)
    )
    return run


PATTERNS = {
    "single": run_single,
    "sequential": run_sequential,
    "supervisor": run_supervisor,
    "parallel": run_parallel,
    "debate": run_debate,
    "debate-diverse": lambda retriever: run_debate(retriever, DIVERSE_PANEL),
}

PATTERN_ORDER = tuple(PATTERNS)

## 9. The comparison

Five orchestrations, one job, one retriever.

In [ ]:
for pattern in PATTERN_ORDER:
    run = PATTERNS[pattern]("keyword-2")
    print(f"{pattern:15s} {len(run.calls):2d} calls  {run.tokens:5d} tokens  "
          f"${run.cost:.5f}  {run.wall_clock():.2f}s  {run.correct}/6 right")

## 10. Where the latency goes

The fan-out runs six workers at once and still takes 3.75 seconds against the
chain's 4.85. Print the schedule and the reason is immediate: the synthesis call
reads all six sections and writes the whole briefing, and it is alone on the
critical path.

In [ ]:
run = run_parallel("keyword-2")
by_id = {call.call_id: call for call in run.calls}
for cid, start, end in sorted(schedule(run.calls)[0], key=lambda b: b[1]):
    call = by_id[cid]
    print(f"{cid:8s} {call.role:8s} {start:5.2f} -> {end:5.2f}  "
          f"prompt={call.prompt_tokens:4d} output={call.output_tokens:3d}")

## 11. The rate limit

Unlimited concurrency is an assumption, and it carries the fan-out's whole case.
Cap the graphs at two calls in flight and the ordering changes.

In [ ]:
print(f"{'pattern':15s}{'uncapped':>10s}{'cap 4':>8s}{'cap 2':>8s}{'cap 1':>8s}")
for pattern in PATTERN_ORDER:
    run = PATTERNS[pattern]("keyword-2")
    print(f"{pattern:15s}{run.wall_clock():>10.2f}{run.wall_clock(4):>8.2f}"
          f"{run.wall_clock(2):>8.2f}{run.wall_clock(1):>8.2f}")

## 12. The chain's context bill

Cut the job into more sections and watch the cost per section move. The chain
re-reads everything it has already retrieved and written; the fan-out does not.

In [ ]:
original = list(SECTIONS)
print(f"{'k':>3s}{'sequential':>12s}{'per section':>13s}{'fan-out':>10s}"
      f"{'per section':>13s}")
for k in range(1, len(original) + 1):
    SECTIONS[:] = original[:k]
    seq, par = run_sequential("keyword-2").tokens, run_parallel("keyword-2").tokens
    print(f"{k:>3d}{seq:>12d}{seq / k:>13.1f}{par:>10d}{par / k:>13.1f}")
SECTIONS[:] = original

## 13. Retrieval against topology

Five retrievers by six orchestrations. The vertical spread is the retriever; the
horizontal spread is the orchestration.

In [ ]:
print(f"{'pattern':16s}" + "".join(f"{r:>11s}" for r in RETRIEVERS))
print("accuracy, sections correct of 6")
for pattern in PATTERN_ORDER:
    cells = "".join(f"{PATTERNS[pattern](r).correct:>11d}" for r in RETRIEVERS)
    print(f"  {pattern:14s}{cells}")

print("\ncost, dollars per briefing")
for pattern in PATTERN_ORDER:
    cells = "".join(f"{PATTERNS[pattern](r).cost:>11.5f}" for r in RETRIEVERS)
    print(f"  {pattern:14s}{cells}")

Every configuration that gets all six sections right, cheapest first. The winner
is one agent making one call against a hand-built section index.

In [ ]:
runs = [PATTERNS[p](r) for p in PATTERN_ORDER for r in RETRIEVERS]
for run in sorted((r for r in runs if r.correct == len(SECTIONS)),
                  key=lambda r: r.cost):
    print(f"{run.pattern:16s}{run.retriever:11s} ${run.cost:.5f}  "
          f"{len(run.calls):2d} calls  {run.wall_clock():.2f}s")

## 14. When a panel is worth four times the calls

Three drafters given the same documents and the same deterministic model produce
the same draft, so the majority is that draft. That is a tautology here, not a
measurement, so model the general case instead: each drafter's latent score is a
shared component plus a private one, mixed by a correlation `rho`, with the
marginal accuracy held fixed so only the correlation changes.

At `rho = 0` the panel is independent and Condorcet's jury theorem applies. At
`rho = 1` the drafters are one drafter. Repeated samples from one model at one
prompt sit near the top of that range.

In [ ]:
import math

import numpy as np
from scipy.special import erfinv

def majority_accuracy(p, n, rho, trials=200_000, seed=20260911):
    rng = np.random.default_rng(seed)
    threshold = -math.sqrt(2) * erfinv(2 * p - 1)
    score = (math.sqrt(rho) * rng.standard_normal(trials)[:, None]
             + math.sqrt(1 - rho) * rng.standard_normal((trials, n)))
    return float(((score > threshold).sum(axis=1) * 2 > n).mean())

for p in (0.6, 0.7, 0.8):
    row = "  ".join(f"rho={rho:.1f}: {majority_accuracy(p, 3, rho):.3f}"
                    for rho in (0.0, 0.5, 0.9, 1.0))
    print(f"one drafter at {p:.0%}, panel of 3   {row}")

## 15. Failure exposure

Cost and accuracy are the two columns people ask for. The third is what a
topology does with a wrong answer once it has one: how many later calls read it,
and whether any of them holds an alternative to compare it against.

In [ ]:
def downstream(run, call):
    reached, frontier = set(), {call.call_id}
    while frontier:
        nxt = {other.call_id for other in run.calls
               if set(other.deps) & frontier and other.call_id not in reached}
        reached |= nxt
        frontier = nxt
    return len(reached)

for pattern in PATTERN_ORDER:
    run = PATTERNS[pattern]("keyword-2")
    producers = [c for c in run.calls
                 if c.role in ("single", "worker", "drafter")]
    reach = [downstream(run, c) for c in producers]
    rivals = 3 if "debate" in pattern else 1
    print(f"{pattern:16s} {sum(reach) / len(reach):.1f} later calls per section,"
          f" {rivals} independent answer(s) at the first reader")

## Exercises

1. **Move the long pole.** The synthesis call holds 2.81 of the fan-out's 3.75
   seconds. Split it into two calls that each write three sections, add a short
   final join, and re-measure. How much of the 2.81 seconds comes back, and what
   does the extra call cost?

2. **Break the router deliberately.** `DOMAIN_CARDS` is derived from each
   domain's documents. Hand-write three cards that separate the domains cleanly
   and re-run `run_supervisor`. Does it reach six of six, and what does that say
   about where the supervisor's accuracy actually lives?

3. **Give the panel a reason to disagree.** `run_debate` takes a `panel` argument.
   Try `("keyword-4", "keyword-8", "index")` against `("keyword-2",) * 3`. Which
   sections change, and what does the cost per corrected section come to?

4. **Add prompt caching.** Charge the sequential chain one tenth of the input
   rate for the prefix each call shares with the one before it. At what number of
   sections does the chain become cheaper than the fan-out?

5. **Widen the job.** Push `SECTIONS` to twelve by splitting the existing six.
   Which pattern's ranking changes first, on cost and on wall-clock, and where
   does the single-call agent stop fitting?

6. **Change the price ratio.** Output tokens cost five times input tokens here.
   Set them equal, then set output at twenty times input. Which orchestration is
   cheapest under each, and why does the ranking move?